<a href="https://colab.research.google.com/github/natdanaiii/Trading/blob/main/Grid_trading_V0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# V0-B Initialized Fixed Grid Backtest

V0-B is a fixed arithmetic **spot grid** prepared for real-trading workflow design.

At startup, the strategy:
- keeps USDT for grid BUY levels that are still below the starting market,
- buys only the BTC quantity required to seed SELL targets above the starting market,
- sizes every seeded BTC position so its BTC quantity matches the quantity that the same grid would receive from a normal future BUY.

This avoids over-sizing the initial BTC inventory.

**Execution assumptions**
- Initial BTC inventory is created at the first candle Open.
- Existing SELL targets are processed before new BUYs in each 1-minute candle.
- BUY occurs only when price crosses a free grid level downward.
- A new downward-crossing BUY cannot SELL in the same candle.
- A level sold in the current candle cannot rebuy in that candle.
- Same-candle SELL proceeds are not reused for BUYs.

This notebook is still a **backtest / shadow-readiness model**. It does not send Binance orders.


## 0. Setup

Imports and Google Drive connection only. Trading logic is kept together in Section 1.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import base64
import bisect
import heapq
import json
import os

import numpy as np
import pandas as pd
import requests


# 1. Trading System

Everything that defines **how the strategy trades** is grouped in this section:
configuration, grid construction, initialization sizing, position economics, and the trading engine.


## 1.1 Trading Configuration

These are the parameters a trader changes.


In [ ]:
# Trading instrument
SYMBOL = "BTCUSDT"

# Trading capital
INITIAL_CAPITAL = 3000.0

# Fixed arithmetic grid
GRID_FLOOR = 38000.0
GRID_CEILING = 127000.0
GRID_GAP = 1000.0

# Trading fees
BUY_FEE = 0.001
SELL_FEE = 0.001


## 1.2 Grid Geometry

A grid slot is:

`BUY at buy_price  →  SELL at sell_target`

The order size is derived later because V0-B initialization depends on the actual starting market price.


In [ ]:
def validate_trading_config(
    capital,
    floor,
    ceiling,
    gap,
    buy_fee,
    sell_fee,
):
    if capital <= 0:
        raise ValueError("INITIAL_CAPITAL must be greater than 0.")
    if floor <= 0:
        raise ValueError("GRID_FLOOR must be greater than 0.")
    if ceiling <= floor:
        raise ValueError("GRID_CEILING must be greater than GRID_FLOOR.")
    if gap <= 0:
        raise ValueError("GRID_GAP must be greater than 0.")
    if not (0 <= buy_fee < 1):
        raise ValueError("BUY_FEE must be in [0, 1).")
    if not (0 <= sell_fee < 1):
        raise ValueError("SELL_FEE must be in [0, 1).")

    raw_grid_count = (ceiling - floor) / gap

    if not np.isclose(raw_grid_count, round(raw_grid_count)):
        raise ValueError(
            "(GRID_CEILING - GRID_FLOOR) must be exactly divisible by GRID_GAP."
        )

    return int(round(raw_grid_count))


def build_grid_template(floor, ceiling, gap):
    buy_prices = np.arange(
        floor,
        ceiling,
        gap,
        dtype=float,
    )

    return pd.DataFrame(
        {
            "grid_id": np.arange(1, len(buy_prices) + 1),
            "buy_price": buy_prices,
            "sell_target": buy_prices + gap,
        }
    )


NUMBER_OF_GRIDS = validate_trading_config(
    INITIAL_CAPITAL,
    GRID_FLOOR,
    GRID_CEILING,
    GRID_GAP,
    BUY_FEE,
    SELL_FEE,
)

GRID_TEMPLATE = build_grid_template(
    GRID_FLOOR,
    GRID_CEILING,
    GRID_GAP,
)

print(f"Number of Grids : {NUMBER_OF_GRIDS}")


## 1.3 Initialized Portfolio Sizing

For a normal future grid BUY at level \(B_i\):

`normal net BTC = Order Size / B_i × (1 - BUY_FEE)`

An initial seeded SELL position must hold **that same net BTC quantity**.

If the starting market price is \(P_0\), the cash required to seed that position is:

`initial cash cost = Order Size × P0 / B_i`

The common normal Order Size is solved so that:

`USDT reserved for future BUY grids + cash used to seed initial BTC = INITIAL_CAPITAL`

This makes initial inventory and later grid cycles use the same grid sizing logic.


In [ ]:
def derive_initialized_grid(
    grid_template,
    start_price,
    initial_capital,
    buy_fee,
):
    grid = grid_template.copy()

    grid_floor = float(grid["buy_price"].min())
    grid_ceiling = float(grid["sell_target"].max())

    if not (grid_floor < start_price < grid_ceiling):
        raise ValueError(
            "Starting market price must be strictly inside the configured grid range."
        )

    # A slot needs BTC at startup if its SELL target is above the market.
    seed_mask = grid["sell_target"] > start_price
    reserve_mask = ~seed_mask

    seed_buy_prices = grid.loc[seed_mask, "buy_price"].to_numpy(float)

    # One unit of normal Order Size requires:
    # - 1.0 USDT weight for each reserved future BUY slot
    # - start_price / grid_buy_price for each seeded BTC slot
    funding_weight = (
        int(reserve_mask.sum())
        + float(np.sum(start_price / seed_buy_prices))
    )

    if funding_weight <= 0:
        raise ValueError("Invalid initialization funding weight.")

    normal_order_size = initial_capital / funding_weight

    grid["order_size_usdt"] = float(normal_order_size)
    grid["seed_at_start"] = seed_mask

    # Net BTC that this grid receives from a normal BUY at its own buy_price.
    grid["normal_net_btc"] = (
        normal_order_size
        / grid["buy_price"]
        * (1.0 - buy_fee)
    )

    # Gross BTC and actual cash needed to create that same net BTC at start_price.
    grid["initial_gross_btc"] = np.where(
        seed_mask,
        grid["normal_net_btc"] / (1.0 - buy_fee),
        0.0,
    )

    grid["initial_entry_cost_usdt"] = np.where(
        seed_mask,
        grid["initial_gross_btc"] * start_price,
        0.0,
    )

    grid["initial_buy_fee_btc"] = np.where(
        seed_mask,
        grid["initial_gross_btc"] * buy_fee,
        0.0,
    )

    reserved_cash = float(
        reserve_mask.sum() * normal_order_size
    )
    seeded_btc_cost = float(
        grid["initial_entry_cost_usdt"].sum()
    )

    if not np.isclose(
        reserved_cash + seeded_btc_cost,
        initial_capital,
        atol=1e-8,
    ):
        raise AssertionError(
            "Initialization sizing does not reconcile to INITIAL_CAPITAL."
        )

    sizing = {
        "normal_order_size_usdt": float(normal_order_size),
        "funding_weight": float(funding_weight),
        "initial_sell_positions": int(seed_mask.sum()),
        "initial_buy_levels": int(reserve_mask.sum()),
        "reserved_cash_usdt": reserved_cash,
        "seeded_btc_cost_usdt": seeded_btc_cost,
    }

    return grid, sizing


## 1.4 Position Model

`Order Size (USDT)` in Trade History means the **actual cash used to open that position**.

- Normal grid BUY: Order Size = the common normal grid order size.
- Initial seeded position: Order Size = the actual cash required at the starting market price to acquire the grid-consistent BTC quantity.


In [ ]:
def create_position_from_normal_buy(
    trade_id,
    grid_index,
    buy_time,
    buy_price,
    sell_target,
    order_size_usdt,
    portfolio_value_at_buy,
    buy_fee,
    sell_fee,
):
    gross_btc = order_size_usdt / buy_price
    buy_fee_btc = gross_btc * buy_fee
    btc_amount = gross_btc - buy_fee_btc

    gross_sell_usdt = btc_amount * sell_target
    sell_fee_usdt = gross_sell_usdt * sell_fee
    net_sell_usdt = gross_sell_usdt - sell_fee_usdt

    return {
        "trade_id": int(trade_id),
        "grid_index": int(grid_index),
        "entry_type": "GRID_BUY",
        "status": "OPEN",
        "buy_time": buy_time,
        "buy_price": float(buy_price),
        "sell_target": float(sell_target),
        "sell_time": pd.NaT,
        "order_size_usdt": float(order_size_usdt),
        "portfolio_value_at_buy": float(portfolio_value_at_buy),
        "order_pct_of_portfolio": float(
            order_size_usdt / portfolio_value_at_buy
        ),
        "portfolio_value_at_sell": np.nan,
        "btc_amount": float(btc_amount),
        "buy_fee_btc": float(buy_fee_btc),
        "sell_fee_usdt": float(sell_fee_usdt),
        "net_sell_usdt": float(net_sell_usdt),
        "net_pnl": np.nan,
    }


def create_seed_position(
    trade_id,
    grid_index,
    buy_time,
    start_price,
    grid_buy_price,
    sell_target,
    normal_order_size_usdt,
    portfolio_value_at_buy,
    buy_fee,
    sell_fee,
):
    # Target net BTC must equal a normal BUY at this grid's own buy price.
    target_net_btc = (
        normal_order_size_usdt
        / grid_buy_price
        * (1.0 - buy_fee)
    )

    gross_btc_at_start = target_net_btc / (1.0 - buy_fee)
    entry_cost_usdt = gross_btc_at_start * start_price
    buy_fee_btc = gross_btc_at_start * buy_fee
    btc_amount = gross_btc_at_start - buy_fee_btc

    gross_sell_usdt = btc_amount * sell_target
    sell_fee_usdt = gross_sell_usdt * sell_fee
    net_sell_usdt = gross_sell_usdt - sell_fee_usdt

    return {
        "trade_id": int(trade_id),
        "grid_index": int(grid_index),
        "entry_type": "INITIAL_SEED",
        "status": "OPEN",
        "buy_time": buy_time,
        "buy_price": float(start_price),
        "grid_buy_price": float(grid_buy_price),
        "sell_target": float(sell_target),
        "sell_time": pd.NaT,
        "order_size_usdt": float(entry_cost_usdt),
        "normal_order_size_usdt": float(normal_order_size_usdt),
        "portfolio_value_at_buy": float(portfolio_value_at_buy),
        "order_pct_of_portfolio": float(
            entry_cost_usdt / portfolio_value_at_buy
        ),
        "portfolio_value_at_sell": np.nan,
        "btc_amount": float(btc_amount),
        "buy_fee_btc": float(buy_fee_btc),
        "sell_fee_usdt": float(sell_fee_usdt),
        "net_sell_usdt": float(net_sell_usdt),
        "net_pnl": np.nan,
    }


## 1.5 Portfolio Initialization


In [ ]:
def initialize_portfolio(
    data,
    grid_template,
    initial_capital,
    buy_fee,
    sell_fee,
):
    start_time = data.iloc[0]["open_time"]
    start_price = float(data.iloc[0]["open"])

    grid, sizing = derive_initialized_grid(
        grid_template,
        start_price,
        initial_capital,
        buy_fee,
    )

    positions = {}
    active_trade_by_grid = {}
    sell_heap = []
    trade_events = []

    cash = float(initial_capital)
    btc = 0.0
    total_buy_fee_usdt = 0.0
    trade_id = 0
    event_id = 0

    seeded_grid_indices = grid.index[
        grid["seed_at_start"]
    ].tolist()

    for grid_index in seeded_grid_indices:
        row = grid.loc[grid_index]

        # Portfolio value immediately before this seed purchase.
        portfolio_value_before = cash + btc * start_price

        trade_id += 1

        position = create_seed_position(
            trade_id=trade_id,
            grid_index=grid_index,
            buy_time=start_time,
            start_price=start_price,
            grid_buy_price=float(row["buy_price"]),
            sell_target=float(row["sell_target"]),
            normal_order_size_usdt=float(
                row["order_size_usdt"]
            ),
            portfolio_value_at_buy=portfolio_value_before,
            buy_fee=buy_fee,
            sell_fee=sell_fee,
        )

        cash_before = cash
        btc_before = btc

        cash -= position["order_size_usdt"]
        btc += position["btc_amount"]

        buy_fee_usdt = (
            position["buy_fee_btc"] * start_price
        )
        total_buy_fee_usdt += buy_fee_usdt

        positions[trade_id] = position
        active_trade_by_grid[grid_index] = trade_id

        heapq.heappush(
            sell_heap,
            (position["sell_target"], trade_id),
        )

        event_id += 1
        trade_events.append(
            {
                "event_id": event_id,
                "time": start_time,
                "side": "BUY",
                "trade_id": trade_id,
                "grid_index": grid_index,
                "price": start_price,
                "cash_movement": -position["order_size_usdt"],
                "grid_cashflow": 0.0,
                "cash_before": cash_before,
                "cash_after": cash,
                "btc_before": btc_before,
                "btc_after": btc,
                "portfolio_value_before": portfolio_value_before,
                "initialization_trade": True,
            }
        )

    if abs(cash) < 1e-10:
        cash = 0.0

    initialization = {
        "start_time": start_time,
        "start_price": float(start_price),
        "normal_order_size_usdt": sizing["normal_order_size_usdt"],
        "funding_weight": sizing["funding_weight"],
        "initial_sell_positions": sizing["initial_sell_positions"],
        "initial_buy_levels": sizing["initial_buy_levels"],
        "initial_cash": float(cash),
        "initial_btc": float(btc),
        "initial_btc_cost_usdt": sizing["seeded_btc_cost_usdt"],
        "reserved_cash_usdt": sizing["reserved_cash_usdt"],
        "initial_buy_fee_usdt": float(total_buy_fee_usdt),
        "initial_btc_allocation_pct": float(
            sizing["seeded_btc_cost_usdt"]
            / initial_capital
            * 100.0
        ),
    }

    return {
        "grid": grid,
        "cash": cash,
        "btc": btc,
        "positions": positions,
        "active_trade_by_grid": active_trade_by_grid,
        "sell_heap": sell_heap,
        "trade_events": trade_events,
        "next_trade_id": trade_id,
        "next_event_id": event_id,
        "initialization": initialization,
        "total_buy_fee_usdt": float(total_buy_fee_usdt),
    }


## 1.6 Trading Engine

For each 1-minute candle:

1. Process SELL targets for positions already open before the BUY phase.
2. Process downward crossings of free BUY grid levels.
3. Mark portfolio value at the candle Close.


In [ ]:
def run_initialized_fixed_grid(
    data,
    grid_template,
    initial_capital,
    buy_fee,
    sell_fee,
):
    data = (
        data
        .sort_values("open_time")
        .reset_index(drop=True)
        .copy()
    )

    if data.empty:
        raise ValueError("Market data is empty.")

    state = initialize_portfolio(
        data,
        grid_template,
        initial_capital,
        buy_fee,
        sell_fee,
    )

    grid = state["grid"]
    cash = state["cash"]
    btc = state["btc"]
    positions = state["positions"]
    active_trade_by_grid = state["active_trade_by_grid"]
    sell_heap = state["sell_heap"]
    trade_events = state["trade_events"]
    trade_id = state["next_trade_id"]
    event_id = state["next_event_id"]
    initialization = state["initialization"]

    normal_order_size = float(
        initialization["normal_order_size_usdt"]
    )

    total_buy_fee_usdt = float(
        state["total_buy_fee_usdt"]
    )
    total_sell_fee_usdt = 0.0
    realized_profit = 0.0
    completed_cycles = 0

    buy_prices = grid["buy_price"].to_numpy(float)
    sell_targets = grid["sell_target"].to_numpy(float)
    buy_price_list = buy_prices.tolist()

    row_count = len(data)
    equity_values = np.empty(row_count)
    cash_values = np.empty(row_count)
    btc_values = np.empty(row_count)

    previous_close = None
    tolerance = 1e-12

    for row_index, candle in enumerate(
        data.itertuples(index=False)
    ):
        timestamp = candle.open_time
        open_price = float(candle.open)
        high_price = float(candle.high)
        low_price = float(candle.low)
        close_price = float(candle.close)

        cash_at_candle_start = cash
        sold_this_candle = set()

        # ----------------------------------------------------
        # 1) Process existing SELL targets
        # ----------------------------------------------------
        while (
            sell_heap
            and sell_heap[0][0] <= high_price + tolerance
        ):
            _, current_trade_id = heapq.heappop(sell_heap)

            position = positions.get(current_trade_id)

            if position is None or position["status"] != "OPEN":
                continue

            grid_index = position["grid_index"]

            cash_before = cash
            btc_before = btc
            portfolio_value_before = (
                cash_before
                + btc_before * position["sell_target"]
            )

            cash += position["net_sell_usdt"]
            btc -= position["btc_amount"]

            if abs(btc) < 1e-12:
                btc = 0.0

            pnl = (
                position["net_sell_usdt"]
                - position["order_size_usdt"]
            )

            position["status"] = "CLOSED"
            position["sell_time"] = timestamp
            position["portfolio_value_at_sell"] = (
                portfolio_value_before
            )
            position["net_pnl"] = float(pnl)

            active_trade_by_grid.pop(grid_index, None)

            total_sell_fee_usdt += position["sell_fee_usdt"]
            realized_profit += pnl
            completed_cycles += 1
            sold_this_candle.add(grid_index)

            event_id += 1
            trade_events.append(
                {
                    "event_id": event_id,
                    "time": timestamp,
                    "side": "SELL",
                    "trade_id": current_trade_id,
                    "grid_index": grid_index,
                    "price": position["sell_target"],
                    "cash_movement": position["net_sell_usdt"],
                    "grid_cashflow": pnl,
                    "cash_before": cash_before,
                    "cash_after": cash,
                    "btc_before": btc_before,
                    "btc_after": btc,
                    "portfolio_value_before": portfolio_value_before,
                    "initialization_trade": False,
                }
            )

        # ----------------------------------------------------
        # 2) Process downward BUY crossings
        # ----------------------------------------------------
        buy_budget = cash_at_candle_start

        downward_start = (
            open_price
            if previous_close is None
            else max(previous_close, open_price)
        )

        if low_price < downward_start:
            first_index = bisect.bisect_left(
                buy_price_list,
                low_price,
            )
            stop_index = bisect.bisect_left(
                buy_price_list,
                downward_start,
            )

            # Higher crossed grid levels are reached first.
            for grid_index in range(
                stop_index - 1,
                first_index - 1,
                -1,
            ):
                if grid_index in active_trade_by_grid:
                    continue
                if grid_index in sold_this_candle:
                    continue

                if (
                    buy_budget + tolerance
                    < normal_order_size
                ):
                    break

                buy_price = float(buy_prices[grid_index])
                sell_target = float(sell_targets[grid_index])

                cash_before = cash
                btc_before = btc
                portfolio_value_before = (
                    cash_before
                    + btc_before * buy_price
                )

                trade_id += 1

                position = create_position_from_normal_buy(
                    trade_id=trade_id,
                    grid_index=grid_index,
                    buy_time=timestamp,
                    buy_price=buy_price,
                    sell_target=sell_target,
                    order_size_usdt=normal_order_size,
                    portfolio_value_at_buy=portfolio_value_before,
                    buy_fee=buy_fee,
                    sell_fee=sell_fee,
                )

                buy_budget -= normal_order_size
                cash -= normal_order_size
                btc += position["btc_amount"]

                total_buy_fee_usdt += (
                    position["buy_fee_btc"]
                    * buy_price
                )

                positions[trade_id] = position
                active_trade_by_grid[grid_index] = trade_id

                heapq.heappush(
                    sell_heap,
                    (sell_target, trade_id),
                )

                event_id += 1
                trade_events.append(
                    {
                        "event_id": event_id,
                        "time": timestamp,
                        "side": "BUY",
                        "trade_id": trade_id,
                        "grid_index": grid_index,
                        "price": buy_price,
                        "cash_movement": -normal_order_size,
                        "grid_cashflow": 0.0,
                        "cash_before": cash_before,
                        "cash_after": cash,
                        "btc_before": btc_before,
                        "btc_after": btc,
                        "portfolio_value_before": portfolio_value_before,
                        "initialization_trade": False,
                    }
                )

        equity = cash + btc * close_price

        equity_values[row_index] = equity
        cash_values[row_index] = cash
        btc_values[row_index] = btc

        previous_close = close_price

    equity_curve = pd.DataFrame(
        {
            "open_time": data["open_time"],
            "close": data["close"],
            "cash": cash_values,
            "btc": btc_values,
            "equity": equity_values,
        }
    )

    trade_event_log = pd.DataFrame(trade_events)

    return {
        "grid": grid,
        "initialization": initialization,
        "positions": positions,
        "trade_event_log": trade_event_log,
        "equity_curve": equity_curve,
        "final_cash": float(cash),
        "final_btc": float(btc),
        "realized_profit": float(realized_profit),
        "completed_cycles": int(completed_cycles),
        "total_buy_fee_usdt": float(total_buy_fee_usdt),
        "total_sell_fee_usdt": float(total_sell_fee_usdt),
    }


# 2. Backtest System

Historical data, evaluation metrics, audit, and the user-facing Trade History live here.


## 2.1 Backtest Configuration


In [ ]:
TIMEFRAME = "1m"
START_DATE = "2024-01-01"
END_DATE = "2026-01-01"

DATA_DIR = "/content/drive/MyDrive/03.Trading/00.Live Trading"


## 2.2 Load Historical Market Data


In [ ]:
def load_market_data(
    symbol,
    timeframe,
    data_dir,
    start_date,
    end_date,
):
    file_path = os.path.join(
        data_dir,
        f"{symbol}-{timeframe}-combined.csv",
    )

    if not os.path.exists(file_path):
        raise FileNotFoundError(file_path)

    data = pd.read_csv(file_path)

    required_columns = {
        "open_time",
        "open",
        "high",
        "low",
        "close",
        "volume",
    }
    missing_columns = required_columns.difference(
        data.columns
    )

    if missing_columns:
        raise ValueError(
            f"Missing columns: {sorted(missing_columns)}"
        )

    data["open_time"] = pd.to_datetime(
        data["open_time"],
        utc=True,
    )

    numeric_columns = [
        "open",
        "high",
        "low",
        "close",
        "volume",
    ]
    data[numeric_columns] = data[numeric_columns].astype(
        float
    )

    start_time = pd.Timestamp(start_date, tz="UTC")
    end_time = pd.Timestamp(end_date, tz="UTC")

    data = (
        data
        .drop_duplicates("open_time")
        .sort_values("open_time")
        .loc[
            lambda df:
            (df["open_time"] >= start_time)
            & (df["open_time"] < end_time)
        ]
        .reset_index(drop=True)
    )

    if data.empty:
        raise ValueError(
            "No market data inside the selected period."
        )

    return data


## 2.3 Performance Statistics


In [ ]:
def performance_stats(
    data,
    equity_curve,
    initial_capital,
):
    equity = equity_curve["equity"].to_numpy(float)

    running_peak = np.maximum.accumulate(equity)
    drawdown = equity / running_peak - 1.0

    final_equity = float(equity[-1])
    max_drawdown = float(drawdown.min())
    net_return = final_equity / initial_capital - 1.0

    elapsed_days = (
        data["open_time"].iloc[-1]
        - data["open_time"].iloc[0]
    ).total_seconds() / 86400.0

    annualized_return = np.nan

    if elapsed_days > 0 and final_equity > 0:
        annualized_log_growth = (
            np.log(final_equity / initial_capital)
            * (365.25 / elapsed_days)
        )

        if annualized_log_growth < 700:
            annualized_return = float(
                np.expm1(annualized_log_growth)
            )

    calmar_ratio = np.nan

    if (
        max_drawdown < 0
        and np.isfinite(annualized_return)
    ):
        calmar_ratio = float(
            annualized_return / abs(max_drawdown)
        )

    equity_curve = equity_curve.copy()
    equity_curve["drawdown"] = drawdown

    return {
        "final_equity": final_equity,
        "net_return": float(net_return),
        "annualized_return": annualized_return,
        "max_drawdown": max_drawdown,
        "calmar_ratio": calmar_ratio,
        "equity_curve": equity_curve,
    }


## 2.4 Trade History

One table for the user. One row = one position / one grid cycle.

For OPEN positions, Net P&L is marked to the final backtest Close.


In [ ]:
def build_trade_history(
    positions,
    final_time,
    final_price,
):
    rows = []

    for trade_id in sorted(positions):
        position = positions[trade_id]

        if position["status"] == "CLOSED":
            sell_time = position["sell_time"]
            portfolio_value_at_sell = (
                position["portfolio_value_at_sell"]
            )
            net_pnl = position["net_pnl"]
            holding_time = sell_time - position["buy_time"]
        else:
            sell_time = pd.NaT
            portfolio_value_at_sell = np.nan

            current_value = (
                position["btc_amount"] * final_price
            )
            net_pnl = (
                current_value
                - position["order_size_usdt"]
            )
            holding_time = final_time - position["buy_time"]

        rows.append(
            {
                "Trade ID": int(position["trade_id"]),
                "Status": position["status"],
                "Buy Time": position["buy_time"],
                "Buy Price": float(position["buy_price"]),
                "Sell Target": float(position["sell_target"]),
                "Sell Time": sell_time,
                "Order Size (USDT)": float(
                    position["order_size_usdt"]
                ),
                "Portfolio Value at Buy": float(
                    position["portfolio_value_at_buy"]
                ),
                "Order % of Portfolio": float(
                    position["order_pct_of_portfolio"]
                    * 100.0
                ),
                "Portfolio Value at Sell": (
                    float(portfolio_value_at_sell)
                    if np.isfinite(portfolio_value_at_sell)
                    else np.nan
                ),
                "Net P&L": float(net_pnl),
                "Holding Time": holding_time,
            }
        )

    return pd.DataFrame(rows)


## 2.5 System Audit

Audit checks accounting plus the V0-B sizing rule:
**every initial seeded position must hold the same net BTC that a normal BUY at that grid level would create.**


In [ ]:
def audit_v0b(
    result,
    trade_history,
    initial_capital,
    buy_fee,
):
    initialization = result["initialization"]
    event_log = result["trade_event_log"]
    equity = result["equity_curve"]
    positions = result["positions"]
    grid = result["grid"]

    cash_movement = (
        float(event_log["cash_movement"].sum())
        if len(event_log)
        else 0.0
    )

    grid_cashflow = (
        float(event_log["grid_cashflow"].sum())
        if len(event_log)
        else 0.0
    )

    open_btc_from_positions = float(
        sum(
            position["btc_amount"]
            for position in positions.values()
            if position["status"] == "OPEN"
        )
    )

    seed_quantity_ok = True

    for position in positions.values():
        if position.get("entry_type") != "INITIAL_SEED":
            continue

        grid_index = position["grid_index"]
        grid_buy_price = float(
            grid.loc[grid_index, "buy_price"]
        )
        normal_order_size = float(
            initialization["normal_order_size_usdt"]
        )

        expected_net_btc = (
            normal_order_size
            / grid_buy_price
            * (1.0 - buy_fee)
        )

        if not np.isclose(
            position["btc_amount"],
            expected_net_btc,
            rtol=0.0,
            atol=1e-12,
        ):
            seed_quantity_ok = False
            break

    closed_trade_count = int(
        trade_history["Status"].eq("CLOSED").sum()
    )

    equity_identity_error = float(
        np.max(
            np.abs(
                equity["cash"]
                + equity["btc"] * equity["close"]
                - equity["equity"]
            )
        )
    )

    checks = {
        "cash_reconciliation": np.isclose(
            initial_capital + cash_movement,
            result["final_cash"],
            atol=1e-8,
        ),
        "realized_profit_reconciliation": np.isclose(
            grid_cashflow,
            result["realized_profit"],
            atol=1e-8,
        ),
        "equity_identity": (
            equity_identity_error <= 1e-8
        ),
        "cash_never_negative": (
            float(equity["cash"].min()) >= -1e-8
        ),
        "initial_allocation_reconciliation": np.isclose(
            initialization["initial_cash"]
            + initialization["initial_btc_cost_usdt"],
            initial_capital,
            atol=1e-8,
        ),
        "all_grid_slots_initialized_or_reserved": (
            initialization["initial_sell_positions"]
            + initialization["initial_buy_levels"]
            == len(grid)
        ),
        "initial_seed_quantity_matches_normal_grid": (
            seed_quantity_ok
        ),
        "final_btc_matches_open_positions": np.isclose(
            result["final_btc"],
            open_btc_from_positions,
            atol=1e-10,
        ),
        "closed_trade_count_reconciliation": (
            closed_trade_count
            == result["completed_cycles"]
        ),
    }

    return checks


## 2.6 Run Backtest & Review Results


In [ ]:
df_1m = load_market_data(
    SYMBOL,
    TIMEFRAME,
    DATA_DIR,
    START_DATE,
    END_DATE,
)

result = run_initialized_fixed_grid(
    data=df_1m,
    grid_template=GRID_TEMPLATE,
    initial_capital=INITIAL_CAPITAL,
    buy_fee=BUY_FEE,
    sell_fee=SELL_FEE,
)

stats = performance_stats(
    df_1m,
    result["equity_curve"],
    INITIAL_CAPITAL,
)
result["equity_curve"] = stats["equity_curve"]

final_time = df_1m["open_time"].iloc[-1]
final_price = float(df_1m["close"].iloc[-1])

trade_history = build_trade_history(
    result["positions"],
    final_time,
    final_price,
)

open_positions = int(
    trade_history["Status"].eq("OPEN").sum()
)

unrealized_pnl = float(
    trade_history.loc[
        trade_history["Status"].eq("OPEN"),
        "Net P&L",
    ].sum()
)

summary = {
    "initial_capital": float(INITIAL_CAPITAL),
    "initial_market_price": float(
        result["initialization"]["start_price"]
    ),
    "normal_order_size_usdt": float(
        result["initialization"]["normal_order_size_usdt"]
    ),
    "initial_cash": float(
        result["initialization"]["initial_cash"]
    ),
    "initial_btc": float(
        result["initialization"]["initial_btc"]
    ),
    "initial_sell_positions": int(
        result["initialization"]["initial_sell_positions"]
    ),
    "initial_buy_levels": int(
        result["initialization"]["initial_buy_levels"]
    ),
    "final_equity": stats["final_equity"],
    "net_return": stats["net_return"],
    "annualized_return": stats["annualized_return"],
    "max_drawdown": stats["max_drawdown"],
    "calmar_ratio": stats["calmar_ratio"],
    "completed_cycles": int(
        result["completed_cycles"]
    ),
    "open_positions": open_positions,
    "final_cash": float(result["final_cash"]),
    "final_btc": float(result["final_btc"]),
    "realized_profit": float(
        result["realized_profit"]
    ),
    "unrealized_pnl": unrealized_pnl,
    "total_fee_usdt_equiv": float(
        result["total_buy_fee_usdt"]
        + result["total_sell_fee_usdt"]
    ),
}

audit_checks = audit_v0b(
    result,
    trade_history,
    INITIAL_CAPITAL,
    BUY_FEE,
)

AUDIT_STATUS = (
    "PASS"
    if all(bool(value) for value in audit_checks.values())
    else "FAIL"
)

print("===== V0-B INITIALIZATION =====")
for key, value in result["initialization"].items():
    print(f"{key:32s}: {value}")

print("\n===== V0-B RESULT =====")
for key, value in summary.items():
    print(f"{key:32s}: {value}")

print("\n===== V0-B AUDIT =====")
for name, passed in audit_checks.items():
    print(
        f"{name:44s}: "
        f"{'PASS' if passed else 'FAIL'}"
    )

print(f"Overall{'':37s}: {AUDIT_STATUS}")

if AUDIT_STATUS != "PASS":
    raise AssertionError("V0-B AUDIT FAILED")


## 2.7 User Trade History

The table below is the single user-facing historical trade table.


In [ ]:
display(trade_history)


# 3. Logging System

Summary/Audit stays in JSON. The user-facing Trade History is exported separately as CSV.


## 3.1 Log Configuration


In [ ]:
REPO = "natdanaiii/Trading"
BRANCH = "main"

GITHUB_SUMMARY_LOG_PATH = (
    "logs/latest_v0_backtest_log.json"
)
GITHUB_TRADE_HISTORY_PATH = (
    "logs/latest_v0_trade_history.csv"
)

LOCAL_SUMMARY_LOG_PATH = (
    "/content/latest_v0_backtest_log.json"
)
LOCAL_TRADE_HISTORY_PATH = (
    "/content/latest_v0_trade_history.csv"
)


## 3.2 Build & Export Logs


In [ ]:
def json_safe(value):
    if value is pd.NaT or value is pd.NA:
        return None
    if isinstance(value, dict):
        return {
            str(key): json_safe(item)
            for key, item in value.items()
        }
    if isinstance(value, (list, tuple)):
        return [json_safe(item) for item in value]
    if isinstance(value, np.ndarray):
        return [
            json_safe(item)
            for item in value.tolist()
        ]
    if isinstance(value, (bool, np.bool_)):
        return bool(value)
    if isinstance(value, (int, np.integer)):
        return int(value)
    if isinstance(value, (float, np.floating)):
        return (
            None
            if not np.isfinite(value)
            else float(value)
        )
    if isinstance(value, pd.Timestamp):
        return value.isoformat()
    return value


def build_summary_log():
    initialization = result["initialization"]

    payload = {
        "log_schema_version": 4,
        "strategy": (
            "V0-B Initialized Fixed Grid "
            "(Grid-Consistent Seed Sizing)"
        ),
        "run_info": {
            "generated_at_utc": (
                pd.Timestamp.now(tz="UTC").isoformat()
            ),
            "repository": REPO,
            "branch": BRANCH,
            "notebook": "Grid_trading_V0.ipynb",
            "symbol": SYMBOL,
            "timeframe": TIMEFRAME,
            "start_date": START_DATE,
            "end_date": END_DATE,
            "data_rows": int(len(df_1m)),
            "data_first_time": (
                df_1m["open_time"].min().isoformat()
            ),
            "data_last_time": (
                df_1m["open_time"].max().isoformat()
            ),
        },
        "trading_config": {
            "initial_capital": INITIAL_CAPITAL,
            "floor": GRID_FLOOR,
            "ceiling": GRID_CEILING,
            "gap": GRID_GAP,
            "buy_fee": BUY_FEE,
            "sell_fee": SELL_FEE,
        },
        "backtest_config": {
            "timeframe": TIMEFRAME,
            "start_date": START_DATE,
            "end_date": END_DATE,
        },
        "derived": {
            "number_of_grids": NUMBER_OF_GRIDS,
            "normal_order_size_usdt": (
                initialization[
                    "normal_order_size_usdt"
                ]
            ),
            "initialization_sizing_method": (
                "Seed each SELL grid with the same net BTC "
                "that a normal BUY at that grid level would create."
            ),
        },
        "initialization": initialization,
        "market_range_diagnostics": {
            "historical_low": float(
                df_1m["low"].min()
            ),
            "historical_high": float(
                df_1m["high"].max()
            ),
            "candles_low_below_floor": int(
                (df_1m["low"] < GRID_FLOOR).sum()
            ),
            "candles_high_above_ceiling": int(
                (
                    df_1m["high"]
                    > GRID_CEILING
                ).sum()
            ),
        },
        "summary": summary,
        "audit": {
            "status": AUDIT_STATUS,
            "checks": audit_checks,
        },
        "trade_history_file": (
            GITHUB_TRADE_HISTORY_PATH
        ),
    }

    return json_safe(payload)


def upload_github_file(
    path,
    content_bytes,
    commit_message,
    github_token,
):
    api_url = (
        f"https://api.github.com/repos/"
        f"{REPO}/contents/{path}"
    )

    headers = {
        "Authorization": f"Bearer {github_token}",
        "Accept": "application/vnd.github+json",
        "X-GitHub-Api-Version": "2022-11-28",
    }

    existing = requests.get(
        api_url,
        headers=headers,
        timeout=30,
    )

    body = {
        "message": commit_message,
        "content": base64.b64encode(
            content_bytes
        ).decode(),
        "branch": BRANCH,
    }

    if existing.status_code == 200:
        body["sha"] = existing.json()["sha"]

    upload = requests.put(
        api_url,
        headers=headers,
        json=body,
        timeout=30,
    )
    upload.raise_for_status()

    return upload.json()["commit"]["sha"]


log_payload = build_summary_log()

with open(
    LOCAL_SUMMARY_LOG_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        log_payload,
        file,
        indent=2,
        allow_nan=False,
    )

trade_history.to_csv(
    LOCAL_TRADE_HISTORY_PATH,
    index=False,
)

try:
    from google.colab import userdata
    github_token = userdata.get("GITHUB_TOKEN")
except Exception:
    github_token = None

if not github_token:
    print(
        "GitHub upload SKIPPED: "
        "Colab Secret 'GITHUB_TOKEN' was not found."
    )
else:
    with open(
        LOCAL_SUMMARY_LOG_PATH,
        "rb",
    ) as file:
        summary_commit = upload_github_file(
            GITHUB_SUMMARY_LOG_PATH,
            file.read(),
            "Update latest V0-B backtest log",
            github_token,
        )

    with open(
        LOCAL_TRADE_HISTORY_PATH,
        "rb",
    ) as file:
        history_commit = upload_github_file(
            GITHUB_TRADE_HISTORY_PATH,
            file.read(),
            "Update latest V0-B trade history",
            github_token,
        )

    print("GitHub log upload: SUCCESS")
    print(
        f"Summary: {GITHUB_SUMMARY_LOG_PATH}"
    )
    print(
        f"Trade History: "
        f"{GITHUB_TRADE_HISTORY_PATH}"
    )
    print(f"Summary Commit: {summary_commit}")
    print(f"History Commit: {history_commit}")
